In [ ]:
!pip install catboost

In [ ]:
!pip install ydata-profiling

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

## Загрузка данных

In [ ]:
!gdown 1ERwQ5odiK1Zvi1LtjpkzCMUswYsAX8_K  # train.csv
!gdown 1fGw_-RFwvn_LEdt91Jq-7A-wzG6mmH8r  # test.csv
!gdown 199Mt4OYZNaelT83U-HGDsEYs2YcUGQ6y  # submission.csv

In [ ]:
data = pd.read_csv('./train.csv')
data

In [ ]:
# Для вашего удобства списки с именами разных колонок

# Числовые признаки
num_cols = [
    'ClientPeriod',
    'MonthlySpending',
    'TotalSpent'
]

# Категориальные признаки
cat_cols = [
    'Sex',
    'IsSeniorCitizen',
    'HasPartner',
    'HasChild',
    'HasPhoneService',
    'HasMultiplePhoneNumbers',
    'HasInternetService',
    'HasOnlineSecurityService',
    'HasOnlineBackup',
    'HasDeviceProtection',
    'HasTechSupportAccess',
    'HasOnlineTV',
    'HasMovieSubscription',
    'HasContractPhone',
    'IsBillingPaperless',
    'PaymentMethod'
]

feature_cols = num_cols + cat_cols
target_col = 'Churn'

In [ ]:
# проверяю наличие NaN-ов:
data.isna().sum()
# Как мы можем увидеть их нет. Это гуд
# НО! ...

In [ ]:
# Если мы посмотрим на:
data.info()
# То увидем у признака TotalSpent тип данных object, что странно ведь это "Траты пользователя"

In [ ]:
# Наличие там NaN-ов можно проверить, если перевести колонку в числовой формат,
# а непреобразуемые значения сделать NaN
data['TotalSpent'] = pd.to_numeric(data['TotalSpent'], errors='coerce')
data.isna().sum()
# Теперь мы можем увидеть, что в колонке TotalSpent действительно есть 9 NaN-ов.

In [ ]:
# Обработаем их, заменив NaN значения на 0, т.к. отсутствие информации в данном случае означает,
# что человек пользуется услугами недавно (ещё не успел ничего оплатить)
data['TotalSpent'] = data['TotalSpent'].fillna(0)
data.isna().sum()

## Анализ данных

In [ ]:
# 1
# Гистограммы для численных признаков
data[num_cols].hist(figsize=(15, 8), bins=60)
plt.show()

In [ ]:
# Из гистограмм выше предполагаю следущее:
# 1. много значений Client Period = 0 => Много новых пользователей (возможно у компании сейчас сильная рекламная программа)
# 2. много значений Client Period = 70 => Много пользователей "старичков"
# 3. много значений Montly Spending = +-20 (видимо самые выгодные тарифы стоят около 20)
# 4. много значений Total Spent = 0 (в связи с большим кол-вом новых пользователей)

In [ ]:
# 1.2
# Вывод кол-ва значений категориальных признаков
for i in cat_cols:
  display(data[i].value_counts())
  print()

In [ ]:
# 1.3(1)
# Графики распределения признаков в категориальных колонках (через plt.bar()):
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(15, 10))
for col_name, ax in zip(cat_cols, axes.flatten()):
  counts = data[col_name].value_counts()
  ax.bar(counts.index, counts.values)
  ax.set_title(col_name)

plt.tight_layout()
plt.show()

In [ ]:
# 1.3(2)
# Графики распределения признаков в категориальных колонках (через plt.pie(), для тренировки):
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(14, 14))
for col_name, ax in zip(cat_cols, axes.flatten()):
  count = data[col_name].value_counts()
  ax.pie(count.values, labels=count.index, autopct='%1.1f%%')
  ax.set_title(col_name)

plt.tight_layout()
plt.show()

In [ ]:
# 2
# Распределение целевой переменной
c = (data[target_col]).value_counts(normalize=True)
plt.bar(c.index, c.values)
plt.title('Распределение целевой переменной')
plt.show()
print('Доли классов:')
display(c)

# Видим, что классы таргета несблансированны

In [ ]:
# 3.1
# Построим график показывающий кто уходит чаще новые пользователи или старые?

sns.boxplot(x=data[target_col], y=data['ClientPeriod'], data=data)

In [ ]:
# 3.2
# Полноценный HTML-отчет, который делает аналитику датасета за нас

from ydata_profiling import ProfileReport
profile = ProfileReport(data, title="Telecom Churn EDA", minimal=True)

profile.to_notebook_iframe()

In [ ]:
# 3.3 Интерактивный Scatter Plot

import plotly.express as px
import plotly.express as px

fig = px.scatter(data,
                 x='ClientPeriod',
                 y='TotalSpent',
                 color='Churn',
                 opacity=0.6,
                 title='Связь срока жизни, трат и оттока')

fig.show()

## Применение линейных моделей

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

In [ ]:
# OHE для категорильальных признаков

data_preproces = pd.get_dummies(data, columns=cat_cols, drop_first=True, dtype=int)
data_preproces.head()

In [ ]:
# разделяем данные
X = data_preproces.drop('Churn', axis=1)
y = data_preproces['Churn']
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42)

In [ ]:
# создаем pipeline
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe_params = {'logisticregression__C': [100, 10, 1, 0.1, 0.01, 0.001]}

In [ ]:
# применяем GridSearchCV внутри pipeline
params = {'C': [100, 10, 1, 0.1, 0.01, 0.001]}

cv_searcher_pipe = GridSearchCV(estimator=pipe,
                          param_grid=pipe_params, cv=5,
                          scoring='roc_auc')

cv_searcher_pipe.fit(X_train, y_train)

print(cv_searcher_pipe.best_params_)
print(cv_searcher_pipe.best_score_)

In [ ]:
# получаем результат
val_pred_pipe = cv_searcher_pipe.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_pred_pipe)
print(roc_auc)

Лучшее качество которое удалось получить: 0.825

## Применение градиентного бустинга

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# разделяем данные
X = data.drop('Churn', axis=1)
y = data['Churn']
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42)

# создаем и обучаем модель
cat_boost_model = CatBoostClassifier(eval_metric='AUC',
                                     random_seed=42,
                                     iterations=500,
                                     learning_rate=0.02,
                                     depth=4,
                                     verbose=False)

cat_boost_model.fit(X_train, y_train, cat_features=cat_cols)

# получаем предсказания и roc-auc
val_pred_cat = cat_boost_model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_pred_cat)
print(roc_auc)

*   Лучшее качество ROC-AUC: 0.828310367695822
*   Параметры: eval_metric='AUC',
                                     random_seed=42,
                                     iterations=500,
                                     learning_rate=0.02,
                                     depth=4,
                                     verbose=False


# Предсказания

In [ ]:
best_model = cat_boost_model

In [ ]:
X_test = pd.read_csv('./test.csv')
submission = pd.read_csv('./submission.csv')

submission['Churn'] = best_model.predict_proba(X_test)[:, 1]
submission.to_csv('./my_submission.csv', index=False)

ИТОГОВЫЙ ROC-AUC на KAGGLE: 0.8523

## Выводы по проделанной работе

### 1) Общие выводы
* **Качество данных**: Исходный датасет содержал скрытые пропуски в колонке `TotalSpent` (пустые строки), которые были успешно преобразованы в 0. Типы данных приведены к корректным (числовым).
* **Анализ данных**: Обнаружен дисбаланс классов (около 26% оттока). Распределения показывают наличие большой группы новых пользователей (ClientPeriod = +-0) и лояльных клиентов (ClientPeriod = +-70). Бльшинство пользователей предпочитают недорогие тарифы (MonthlySpending = +-20).
* **Зависимости**: Визуальный анализ подтвердил, что чаще уходят новые клиенты с коротким жизненным циклом.

### 2) Выбор и оценка модели
Были протестированы два подхода:
1. **Логистическая регрессия**: С использованием StandardScaler и OneHotEncoding. Достигнут ROC-AUC ~0.825 на валидации.
2. **CatBoostClassifier**: Градиентный бустинг показал лучший результат (ROC-AUC = 0.828 на валидации и 0.8523 на тесте Kaggle) благодаря встроенной обработке категориальных признаков и способности улавливать нелинейные зависимости.

### 3) Feature Importance
Исходя из анализа и характера модели, наиболее важными признаками для прогнозирования оттока являются:
* **ClientPeriod**: Длительность отношений с компанией (чем меньше срок, тем выше риск).
* **MonthlySpending** и **TotalSpent**: Финансовые показатели.
* **HasContractPhone**: Тип контракта (месячный контракт сильнее коррелирует с оттоком, чем годовой).
* **PaymentMethod**: Способ оплаты (электронные чеки часто связаны с более высоким уровнем оттока).

### 4) Короткое резюме для бизнеса
Снижению оттока помогут помочь:
1. **Фокус на новичках**: Разработать программу адаптации для клиентов в первые 1-5 месяцев использования сервиса.
2. **Удержание через контракты**: Стимулировать переход с месячной оплаты на долгосрочные контракты (на 1-2 года).
3. **Работа с тарифами**: Проверить удовлетворенность клиентов, использующих Fiber Optic и Electronic Check, так как в этих сегментах наблюдается повышенная склонность к уходу.